In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import leaguedashteamstats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


def fetch_team_data(seasons=None):
    """Fetch NBA team data from multiple seasons."""
    if seasons is None:
        seasons = [
            '2019-20',
            '2020-21',
            '2021-22',
            '2022-23',
            '2023-24',
            '2024-25'
        ]

    print(f"Fetching NBA team data for {len(seasons)} seasons...")

    all_dfs = []

    for season in seasons:
        print(f"  -> Fetching {season}")
        team_stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced'
        )
        df = team_stats.get_data_frames()[0]
        df['SEASON'] = season  # 👈 track season
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Total rows collected: {len(combined_df)}")

    return combined_df


def prepare_data(df):
    """Prepare features and create binary target from WIN_TIER."""
    features = [
        'OFF_RATING',
        'DEF_RATING',
        'NET_RATING',
        'PACE',
        'TS_PCT',
        'EFG_PCT',
        'AST_RATIO',
        'REB_PCT',
        'TM_TOV_PCT',
        'PIE'
    ]

    df_clean = df[['TEAM_NAME'] + features + ['W_PCT']].dropna()
    df_clean['WIN_TIER'] = pd.qcut(
        df_clean['W_PCT'],
        3,
        labels=['Low', 'Mid', 'High'],
        duplicates='drop'
    )

    print("Original class distribution:")
    print(df_clean['WIN_TIER'].value_counts().sort_index())

    class_counts = df_clean['WIN_TIER'].value_counts()
    top_2_classes = class_counts.nlargest(2).index.tolist()
    dropped_class = class_counts.index.difference(top_2_classes).tolist()

    df_binary = df_clean[df_clean['WIN_TIER'].isin(top_2_classes)].copy()
    df_binary['BINARY_TARGET'] = df_binary['WIN_TIER'].map(
        {top_2_classes[0]: 0, top_2_classes[1]: 1}
    )

    print(f"\nFiltered to 2 classes: {top_2_classes}")
    print(f"Dropped class: {dropped_class[0]} ({class_counts[dropped_class[0]]} teams)")
    print(f"Binary distribution: {df_binary['BINARY_TARGET'].value_counts().sort_index()}")

    X = df_binary[features].copy()
    y = df_binary['BINARY_TARGET']
    class_names = top_2_classes 

    return X, y, class_names, features


def plot_side_by_side_confusion_matrices(
    y_true, y_pred_lr, y_pred_nb,
    class_names, filename
):
    """Plot and save side-by-side confusion matrices for LR and NB."""
    cm_lr = confusion_matrix(y_true, y_pred_lr)
    cm_nb = confusion_matrix(y_true, y_pred_nb)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sns.heatmap(
        cm_lr,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0],
        cbar_kws={'shrink': 0.8}
    )
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title('Logistic Regression - Confusion Matrix')

    sns.heatmap(
        cm_nb,
        annot=True,
        fmt='d',
        cmap='Greens',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[1],
        cbar_kws={'shrink': 0.8}
    )
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title('Multinomial NB - Confusion Matrix')

    plt.suptitle('Model Comparison: Logistic Regression vs Multinomial NB', fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


def main():
    df = fetch_team_data()
    X, y, class_names, feature_names = prepare_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\n=== Logistic Regression ===")
    lr_clf = LogisticRegression(random_state=42, max_iter=1000)
    lr_clf.fit(X_train_scaled, y_train)
    y_pred_lr = lr_clf.predict(X_test_scaled)

    print(classification_report(y_test, y_pred_lr, target_names=class_names))

    print("\n=== Multinomial Naive Bayes ===")
    X_train_nb = np.abs(X_train_scaled)
    X_test_nb = np.abs(X_test_scaled)

    nb_clf = MultinomialNB()
    nb_clf.fit(X_train_nb, y_train)
    y_pred_nb = nb_clf.predict(X_test_nb)

    print(classification_report(y_test, y_pred_nb, target_names=class_names))

    plot_side_by_side_confusion_matrices(
        y_test, y_pred_lr, y_pred_nb,
        class_names,
        'outputs/logistic_regression_vs_nb_comparison.png'
    )

    print("\n=== Summary ===")
    print(f"Filtered data to 2 classes: {class_names}")
    print("Side-by-side comparison saved to: outputs/logistic_regression_vs_nb_comparison.png")
    print("\nNote: Multinomial NB uses absolute values of standardized features")
    print("to ensure non-negative input as required by the algorithm.")


if __name__ == "__main__":
    main()


Fetching NBA team data for 6 seasons...
  -> Fetching 2019-20
  -> Fetching 2020-21
  -> Fetching 2021-22
  -> Fetching 2022-23
  -> Fetching 2023-24
  -> Fetching 2024-25
Total rows collected: 180
Original class distribution:
WIN_TIER
Low     60
Mid     66
High    54
Name: count, dtype: int64

Filtered to 2 classes: ['Mid', 'Low']
Dropped class: High (54 teams)
Binary distribution: BINARY_TARGET
0.0    66
1.0    60
Name: count, dtype: int64

=== Logistic Regression ===
              precision    recall  f1-score   support

         Mid       0.87      0.93      0.90        14
         Low       0.91      0.83      0.87        12

    accuracy                           0.88        26
   macro avg       0.89      0.88      0.88        26
weighted avg       0.89      0.88      0.88        26


=== Multinomial Naive Bayes ===
              precision    recall  f1-score   support

         Mid       0.67      0.71      0.69        14
         Low       0.64      0.58      0.61        12

 